# TipCat Pipeline Manager — Phone Cases
Interactive notebook for managing phone case designs and running pipeline steps.

In [ ]:
import json
import os
import subprocess
from google.cloud import storage
from datetime import datetime
from pathlib import Path

# Configuration
CONFIG_NAME = "tipcat-phonecases"
GCP_PROJECT = "tipcat-automation"
GCS_BUCKET = "tipcat-product-designs"
CLOUD_RUN_JOB = "tipcat-phonecases-pipeline"
CLOUD_RUN_REGION = "us-central1"

storage_client = storage.Client(project=GCP_PROJECT)
bucket = storage_client.bucket(GCS_BUCKET)
print(f"✓ Connected to GCP project: {GCP_PROJECT}")
print(f"✓ GCS bucket: {GCS_BUCKET}")
print(f"✓ Config: {CONFIG_NAME}")
print(f"✓ Cloud Run job: {CLOUD_RUN_JOB}")

In [ ]:
def sync_local_designs_to_gcs(local_dir, archive_existing=True, clear_existing=True, include_subdirs=False):
    """Upload local PNGs to gs://<bucket>/designs/ from notebook runtime."""
    local_path = Path(local_dir).expanduser().resolve()
    if not local_path.exists() or not local_path.is_dir():
        raise ValueError(f"Local directory not found: {local_path}")

    pattern = '**/*.png' if include_subdirs else '*.png'
    local_files = sorted(local_path.glob(pattern))
    local_files = [p for p in local_files if p.is_file()]

    if not local_files:
        raise ValueError(f"No PNG files found in: {local_path}")

    existing_design_blobs = [
        b for b in bucket.list_blobs(prefix='designs/')
        if b.name.lower().endswith('.png') and b.name.count('/') == 1
    ]

    print(f"Local PNGs found: {len(local_files)}")
    print(f"Current GCS top-level designs: {len(existing_design_blobs)}")

    if archive_existing and existing_design_blobs:
        ts = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
        archive_prefix = f"archive/designs-backup-{ts}/"
        print(f"\n📦 Archiving existing designs to gs://{GCS_BUCKET}/{archive_prefix}")
        for blob in existing_design_blobs:
            target_name = archive_prefix + blob.name.split('/')[-1]
            bucket.copy_blob(blob, bucket, new_name=target_name)

    if clear_existing and existing_design_blobs:
        print(f"🧹 Deleting {len(existing_design_blobs)} existing designs from gs://{GCS_BUCKET}/designs/")
        for blob in existing_design_blobs:
            blob.delete()

    print(f"\n☁️ Uploading {len(local_files)} local PNGs...")
    uploaded = []
    for f in local_files:
        dest_name = f"designs/{f.name}"
        blob = bucket.blob(dest_name)
        blob.upload_from_filename(str(f))
        uploaded.append(dest_name)

    print(f"✅ Upload complete: {len(uploaded)} files")
    print("Run refresh_inventory() next to regenerate product_list.json")
    return uploaded

# Example:
# sync_local_designs_to_gcs('/content/designs', archive_existing=True, clear_existing=True)

In [ ]:
def list_designs(prefix="designs/"):
    """List all PNG designs in GCS bucket."""
    blobs = list(bucket.list_blobs(prefix=prefix))
    designs = [
        b.name for b in blobs
        if b.name.lower().endswith(".png") and b.name.count("/") == 1
    ]
    designs = sorted(designs)
    print(f"\n📁 Found {len(designs)} PNG designs:")
    for name in designs[:20]:
        print(f"  - {name}")
    if len(designs) > 20:
        print(f"  ... and {len(designs) - 20} more")
    return designs

print("\n=== Scanning for designs ===")
designs = list_designs()

In [ ]:
def refresh_inventory():
    """Scan GCS for designs and update product list JSON."""
    designs = list_designs()
    
    # Create product list from design filenames
    products = []
    for i, design_path in enumerate(designs, 1):
        filename = design_path.split('/')[-1].replace('.png', '')
        products.append({
            "sku": str(i),
            "design_name": filename,
            "gcs_path": design_path,
            "status": "pending"
        })
    
    # Write product list to GCS
    product_list = {
        "config": CONFIG_NAME,
        "generated_at": datetime.now().isoformat(),
        "total_designs": len(products),
        "products": products
    }
    
    blob = bucket.blob("output/product_list.json")
    blob.upload_from_string(
        json.dumps(product_list, indent=2),
        content_type="application/json"
    )
    
    print(f"\n✓ Updated product_list.json: {len(products)} designs")
    return product_list

# Refresh on startup
product_list = refresh_inventory()

In [ ]:
def run_step(step=1, limit=None, verbose=True, config=CONFIG_NAME):
    """Execute a single pipeline step via Cloud Run."""
    args = [
        f"--config={config}",
        f"--step={step}"
    ]
    
    if limit and int(limit) > 0:
        args.append(f"--limit={int(limit)}")
    
    if verbose:
        args.append("--verbose")
    
    cmd = [
        "gcloud", "run", "jobs", "execute", CLOUD_RUN_JOB,
        f"--region={CLOUD_RUN_REGION}",
        f"--project={GCP_PROJECT}",
        f"--args={','.join(args)}"
    ]
    
    print(f"\n🚀 Running Step {step}...")
    print(f"Command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"Exit code: {result.returncode}")
    
    if result.stdout:
        print("\n📋 Output:")
        print(result.stdout[:2000])
    
    if result.stderr:
        print("\n❌ Errors:")
        print(result.stderr[:2000])
    
    if result.returncode == 0:
        print(f"\n✓ Step {step} completed successfully")
    
    return result

# Example usage:
# result = run_step(step=1, limit=2, verbose=True)

In [ ]:
def read_generated_metadata():
    """Read and display generated metadata from Step 1."""
    try:
        blob = bucket.blob("output/generated_metadata.json")
        data = json.loads(blob.download_as_text())
        print(f"\n📊 Generated Metadata: {len(data)} items")
        
        for item in data[:3]:
            sku = item.get("sku")
            title = item.get("analysis", {}).get("metadata", {}).get("title", "")
            status = item.get("analysis", {}).get("status", "")
            print(f"\n  SKU {sku}: {title}")
            print(f"    Status: {status}")
        
        if len(data) > 3:
            print(f"\n  ... and {len(data) - 3} more items")
        
        return data
    except Exception as e:
        print(f"Error reading metadata: {e}")
        return None

# Example:
# metadata = read_generated_metadata()